# Test Synthethic Data Generation (SDG) AUG-PE Algorithm

The idea of this algorithm is to generate synthetic copies of sensitive text data using an LLM with an API without the LLM ever reading any of the real data. The algorithm also includes a Differential Privacy (DP) garuantee that ensure that the removal of any one record does not impact the distribution of the data, strenghthening the privacy protections of the algorithm. 

I think in a proper implementation I would set up two classes, one that included the public data and api access and one that had the private data in a hidden variable just to avoid an overlap.

## Step 0 - Input Variables

Generate some fake restuarant reviews for testing and some of the global AUG-PE parameters.

In [ ]:
import anthropic
import os
import numpy as np
import pandas as pd
from tabulate import tabulate
from sentence_transformers import SentenceTransformer

In [ ]:
# Restaurant reviews
private_data = [
    "The restaurant was fantastic and the service was quick.",
    "Food arrived cold and delivery was very slow.",
    "The pizza crust was so good! But the topping's were a bit disappointing."
]

# Global Varibales
EPSILON = 1.0   # DP privacy budget (lower = more private)
N_SYNTHETIC = 3
N_CANDIDATES = 6  # generate more candidates so histogram has real competition
T = 3  # number of AUG-PE iterations


In [12]:
# Connect to anthropic API and load the encoding model
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
encoder = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4751.21it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Step 1 - DP_NN_HISTOGRAM

DP_NN_HISTOGRAM constructs a histogram of the nearest neighbours of the real data compared to the synthethic data. Distance is based on the embedding model $\Phi$. Then random gaussian noise $\frac{1}{\epsilon}$ is added to enforce DP privacy. This histogram is then used to draw samples from.

At the most basic level, it is just counting for each synthethic record how many real records is the synthetic record the most similar to. 

In [13]:
# --- DP-NN HISTOGRAM ---
def dp_nn_histogram(candidates, private_texts, epsilon):
    """
    1. Each private sample votes for its nearest candidate (no noise on embeddings)
    2. Laplace noise added to the vote counts (the DP mechanism)
    3. Candidate with highest noisy count wins
    """
    priv_embs = encoder.encode(private_texts, normalize_embeddings=True)
    cand_embs = encoder.encode(candidates, normalize_embeddings=True)

    # Build raw vote histogram
    counts = np.zeros(len(candidates))
    print("\n  [Votes]")
    for i, emb in enumerate(priv_embs):
        best_idx = int(np.argmax(cand_embs @ emb))
        counts[best_idx] += 1
        print(f"    Private [{i+1}] → Candidate [{best_idx+1}]: {candidates[best_idx]}")

    print(f"\n  [Raw counts]   {counts.astype(int).tolist()}")

    # Add Laplace noise to histogram (sensitivity=1 since each person casts 1 vote)
    noisy_counts = counts + np.random.laplace(0, 1.0 / epsilon, size=counts.shape)
    print(f"  [Noisy counts] {[f'{v:.2f}' for v in noisy_counts]}")

    winner_idx = int(np.argmax(noisy_counts))
    print(f"  [Winner] Candidate [{winner_idx+1}]: {candidates[winner_idx]}")
    return candidates[winner_idx]

# Step 2 - RANDOM_API

The random api uses a prompt to draw initial candidates from the LLM API. In future, some additional subcategories should be enforced for better initital draws. 

In [14]:
# --- RANDOM API ---
def random_api(n):
    print(f"\n[RANDOM API] Generating {n} initial candidates...")
    prompt = (
        f"Generate {n} diverse, realistic restaurant reviews covering different sentiments. "
        "Each review should be 1-2 sentences. "
        "Return ONLY a numbered list, one review per line."
    )
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )
    lines = [l.strip() for l in response.content[0].text.strip().split("\n") if l.strip()]
    candidates = []
    for line in lines:
        parts = line.split('. ', 1)
        candidates.append(parts[1].strip() if len(parts) == 2 and parts[0].isdigit() else line)
    candidates = candidates[:n]
    for i, c in enumerate(candidates, 1):
        print(f"  {i}. {c}")
    return candidates

## Step 3 - VARIATION API

The VARIATION_API is responsible for drawing a sample from the DP_NN_HISTOGRAM, and then using the LLM API to add some variation to the drawn set of samples. This just helps to add some additional variation to the samples.



In [15]:
# --- VARIATION API ---
def variation_api(text):
    prompt = (
        "Rewrite the following restaurant review in a different style, "
        "keeping the same general sentiment. Return only the rewritten review, no explanation.\n\n"
        f"Review: {text}"
    )
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=256,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()


## Step 4 - Run the Loop

Generates random reviews, finds the most similar records, adds variation to them, normally repeats for a certain number of runs. 

TODO - need to actually loop through multiple times

In [ ]:
# Draw initial candidates
candidates = random_api(N_CANDIDATES)
iteration_results = []

# Loop over the histogram and variation api
for t in range(T):
    print(f"\n{'='*60}")
    print(f"ITERATION T={t+1}")
    print(f"{'='*60}")

    print("\n[DP-NN HISTOGRAM]")
    winner = dp_nn_histogram(candidates, private_data, EPSILON)
    iteration_results.append(winner)

    if t < T - 1:
        print(f"\n[VARIATION API] Generating {N_CANDIDATES} new candidates from winner...")
        candidates = variation_api(winner, N_CANDIDATES)
        for i, c in enumerate(candidates, 1):
            print(f"  {i}. {c}")


[RANDOM API] Generating 6 initial candidates...
  1. The pasta was overcooked and the service was incredibly slow - we waited 45 minutes just for our appetizers to arrive.
  2. Absolutely phenomenal dining experience with perfectly seasoned steaks and attentive staff who made our anniversary dinner truly special.
  3. The food was decent and reasonably priced, though nothing particularly stood out as memorable.
  4. While the ambiance is lovely and the cocktails are creative, the entrees were disappointingly bland and overpriced for the portion sizes.
  5. This hidden gem serves the most authentic Thai cuisine I've had outside of Bangkok, with bold flavors and fresh ingredients in every dish.
  6. The restaurant has potential with its great location and friendly servers, but the kitchen needs work - my salmon was dry and the vegetables were undercooked.

[DP-NN HISTOGRAM] Voting across all 6 candidates (ε=1.0)...

  [Votes]
    Private [1] → Candidate [1]: The pasta was overcooked and

## Step 5 - Display results

In [ ]:
# --- RESULTS TABLE ---
def render_table(private_data, iteration_results):
    real_embs = encoder.encode(private_data, normalize_embeddings=True)
    rows = []
    for t, winner in enumerate(iteration_results):
        syn_emb = encoder.encode([winner], normalize_embeddings=True)[0]
        sims = [float(np.dot(r, syn_emb)) for r in real_embs]
        best_match_idx = int(np.argmax(sims))
        best_sim = sims[best_match_idx]
        conf = "High" if best_sim >= 0.8 else "Medium" if best_sim >= 0.6 else "Low"
        rows.append([f"T={t+1}", private_data[best_match_idx][:60], winner[:60], f"{best_sim:.3f}", conf])

    print(tabulate(rows, headers=["Iter", "Closest Real Match", "Synthetic Winner", "Cosine Sim", "Confidence"], tablefmt="rounded_outline", maxcolwidths=[6, 50, 50, 10, 10]))
    print(f"\nDP ε={EPSILON} — Total budget spent: {EPSILON * T:.2f} ({T} iterations)")

render_table(private_data, iteration_results)


RESULTS  (DP ε=1.0)
 #                                                              Real Review                                                                                                                                                                                                                 Synthetic Review Cosine Sim
 1                  The restaurant was fantastic and the service was quick. While the cuisine didn't blow me away, it hit the spot for a laid-back dinner out. The bill didn't break the bank, and the vibe was cozy and relaxed - perfect for catching up with friends over a solid, if unremarkable, meal.      0.559
 2                            Food arrived cold and delivery was very slow.        While this spot won't blow you away with culinary innovation, it delivers solid comfort food at wallet-friendly prices, making it a reliable choice when you're craving a relaxed dining experience without any pretense.      0.156
 3 The pizza crust was so good! But the top